# Analysis of the image-regression experiments

This notebook reads the outputs of `main_run.py` in `./checkpoints` and

1. produces the result tables and the energy figure of the paper (`evaluate.py`);
2. compares, grade by grade, the Sobolev energy of each MGDL model on $\Omega$ with its energy at the training pixels;
3. checks that the compared runs were trained with the same settings.

Run it from the folder that contains `evaluate.py` and `checkpoints/`.

## 1. Result tables and energy figure

Writes the tables as CSV files and the figure `mgdl_energy.pdf` to `./results`.

In [ ]:
%run evaluate.py

## 2. Energy on $\Omega$ and at the training pixels, grade by grade

For $F=(F_1,F_2,F_3)$ and $r=1,2$, with $\|\nabla^rF(\mathbf x)\|_F^2$ the sum of the squares of all derivatives of order $r$ of the three channels:

* energy on the domain, estimated by Monte Carlo from uniform points of $\Omega=[-1,1]^2$ (131,072 points for $r=1$, 32,768 for $r=2$):
$$\mathcal O^{(r)}_\Omega(F)=\int_\Omega\|\nabla^rF(\mathbf x)\|_F^2\,d\mathbf x ;$$
* energy at the training pixels, the same integral computed by one point per cell, on a fixed subset of 32,768 training pixels:
$$\mathcal O^{(r)}_{\mathcal D_{\rm train}}(F)=\frac{|\Omega|}{|\mathcal D_{\rm train}|}\sum_{\mathbf x\in\mathcal D_{\rm train}}\|\nabla^rF(\mathbf x)\|_F^2 ;$$
* their ratio
$$\text{ratio}_r(F_k)=\mathcal O^{(r)}_\Omega(F_k)\,/\,\mathcal O^{(r)}_{\mathcal D_{\rm train}}(F_k).$$

**Reading.** ratio $\approx 1$: the derivatives are as large at the training pixels as between them;
ratio $\gg 1$: the model is flat at the training pixels and steep between them;
ratio $\ll 1$: the derivatives are concentrated at the training pixels.

Both energies are stored in the history of every run (`O_Fk`, `O_grid_Fk`, `O2_Fk`, `O2_grid_Fk`), so nothing is recomputed here.

In [ ]:
import json
import os

import pandas as pd

CHECKPOINT_DIR = "./checkpoints"
IMAGE = "santorini"                 # "santorini", "tokyo" or "sst"

MODELS = {                          # label -> method in the run name
    "no-reg": "noreg",
    "weight decay": "WD",
    "H1 at train": "H1_train",
    "H1 on Omega": "H1",
    "H2 at train": "H2_train",
    "H2 on Omega": "H2",
}


def load(run):
    path = os.path.join(CHECKPOINT_DIR, f"{run}_history.json")
    if not os.path.exists(path):
        print(f"(missing: {run})")
        return None
    with open(path) as fh:
        return json.load(fh)


rows = []
for label, method in MODELS.items():
    blob = load(f"{IMAGE}_mgdl_{method}")
    if blob is None:
        continue
    for h in blob["history"]:
        rows.append({
            "model": label, "grade": int(h["grade"]),
            "train PSNR": h["psnr_train_dB"], "unseen PSNR": h["psnr_unseen_dB"],
            "test PSNR": h["psnr_test_dB"],
            "O1 Omega": h["O_Fk"], "O1 train": h["O_grid_Fk"],
            "ratio_1": h["O_Fk"] / h["O_grid_Fk"],
            "O2 Omega": h["O2_Fk"], "O2 train": h["O2_grid_Fk"],
            "ratio_2": h["O2_Fk"] / h["O2_grid_Fk"],
        })

df = pd.DataFrame(rows)
df["model"] = pd.Categorical(df["model"], categories=list(MODELS), ordered=True)
df = df.sort_values(["model", "grade"]).reset_index(drop=True)
os.makedirs("results", exist_ok=True)
df.to_csv(f"results/energy_ratios_{IMAGE}.csv", index=False)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.width", 200)

In [ ]:
# ratio_1 = O^(1)_Omega / O^(1)_train   (rows: grade, columns: model)
df.pivot_table(index="grade", columns="model", values="ratio_1", observed=True)

In [ ]:
# ratio_2 = O^(2)_Omega / O^(2)_train
df.pivot_table(index="grade", columns="model", values="ratio_2", observed=True)

In [ ]:
# PSNR per grade; "unseen" = pixels used neither for training nor for selecting kappa
df.pivot_table(index="grade", columns="model",
               values=["train PSNR", "unseen PSNR", "test PSNR"], observed=True)

## 3. Were the compared runs trained with the same settings?

Compares the training arguments stored with each run against a reference run. Only the arguments that are meant to differ between the methods may differ; anything else means the runs are not comparable.

In [ ]:
REFERENCE = "H1 on Omega"
EXPECTED = {"model_name", "reg_order", "kappa", "reg_type", "penalty_points"}

args = {}
for label, method in MODELS.items():
    blob = load(f"{IMAGE}_mgdl_{method}")
    if blob is not None:
        args[label] = blob["args"]
ref = args[REFERENCE]
for label, a in args.items():
    bad = {k: (ref.get(k), a.get(k)) for k in sorted(set(ref) | set(a))
           if ref.get(k) != a.get(k) and k not in EXPECTED}
    print(f"{label:14s} {'OK' if not bad else 'DIFFERENT SETTINGS'}")
    for k, (r, v) in bad.items():
        print(f"      {k}: {REFERENCE} = {r!r}, this run = {v!r}")